<a href="https://colab.research.google.com/github/amiralito/RFdiffusion3_Colab/blob/main/RFdiffusion3_design_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFdiffusion3 design pipeline (Foundry)

General-purpose **RFD3 → LigandMPNN → RF3** pipeline on Colab. The input JSON builder is
parameterised, so the same notebook covers unconditional generation, monomer/motif
scaffolding, binder design, and **Cn/Dn symmetric** design — set the parameters for the run
you want.

**Stack** — [RosettaCommons/foundry](https://github.com/RosettaCommons/foundry):
`rfd3` (all-atom diffusion) → `mpnn` (LigandMPNN inverse folding) → `rf3` (open AF3-class validator).

**Before you run anything**
- `Runtime → Change runtime type → GPU`. RFD3 + RF3 are heavy. A **T4 (16 GB) is the floor**
  (use `low_memory_mode`); an **L4 / A100** is much more comfortable. Symmetric runs are
  forced to `diffusion_batch_size=1`, so watch VRAM on large symmetric targets.
- Install + weights take a while (multi-GB). Do them once per session.

**Scope note.** Install, MPNN, and RF3 follow the documented CLI and are mechanical. The
parts that need empirical iteration are the **contig + hotspot** spec for your target and —
if you go symmetric — preparing an **exactly Cn/Dn, origin-centred, axis-aligned** target
(step 5). The JSON builder prints the exact spec so you can hand-edit it.


## 1 · Check the GPU

In [ ]:
#@title Check allocated GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU — set Runtime → Change runtime type → GPU"


## Setup · Output directory & Google Drive

Sets one **`WORK`** directory that every downstream step reads/writes. Leave it at
`/content/run` for ephemeral local storage, or mount Drive and point `OUTPUT_DIR` at a Drive
path to **persist a whole run** (backbones, sequences, refolds, metrics) across sessions.

Optionally point the checkpoint dir at Drive too (commented below) so the multi-GB weights
survive session resets and you skip re-downloading in step 3.


In [ ]:
#@title Mount Drive & set output directory
import os
MOUNT_DRIVE = False           #@param {type:"boolean"}
OUTPUT_DIR  = "/content/run"  #@param {type:"string"}
# To persist: tick MOUNT_DRIVE and set OUTPUT_DIR to e.g.
#   /content/drive/MyDrive/rfd3_runs/nrc2_cap

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # Optional: persist weights on Drive too (saves re-downloading in step 3):
    # os.environ["FOUNDRY_CHECKPOINT_DIRS"] = "/content/drive/MyDrive/foundry_ckpts"

WORK = OUTPUT_DIR
os.makedirs(WORK, exist_ok=True)
print("Output dir (WORK):", WORK)
if MOUNT_DRIVE:
    print("Drive mounted:", os.path.isdir("/content/drive/MyDrive"))


## 2 · Install Foundry

`rc-foundry[all]` pulls RFD3, LigandMPNN, RF3 and the shared AtomWorks stack. This is the
slow cell. If the PyPI install hits a resolver/CUDA snag on Colab, fall back to the git
install in the commented block.


In [ ]:
#@title Install rc-foundry  (~several minutes)
import time, os
t0 = time.time()

# Primary path: PyPI
!pip install -q "rc-foundry[all]"

# --- Fallback (uncomment if the above fails) ----------------------------------
# !git clone -q https://github.com/RosettaCommons/foundry.git
# !pip install -q -e "foundry/.[all]"
# ------------------------------------------------------------------------------

# Light deps used by the prep/analysis cells below
!pip install -q biotite pandas

print(f"\\nInstall finished in {time.time()-t0:.0f}s")
# Sanity check the CLIs are on PATH
!which rfd3 mpnn rf3 foundry || true


## 3 · Download model weights

Weights land in `~/.foundry/checkpoints` by default; Foundry searches there automatically
at inference time, so no checkpoint paths are needed downstream.


In [ ]:
#@title Download RFD3 + LigandMPNN + RF3 checkpoints
import os
CKPT_DIR = "/content/foundry_ckpts"  #@param {type:"string"}
os.environ["FOUNDRY_CHECKPOINT_DIRS"] = CKPT_DIR
os.makedirs(CKPT_DIR, exist_ok=True)

!foundry install rfd3 ligandmpnn rf3 --checkpoint-dir "$CKPT_DIR"
# List what's available / downloaded:
!foundry install --help 2>/dev/null | head -n 20 || true
print("\\nCheckpoints in", CKPT_DIR)
!ls -lh "$CKPT_DIR" 2>/dev/null || true


## 4 · Provide the target structure *(optional)*

Skip this for unconditional generation. Otherwise upload a PDB/CIF, **or** fetch one by
accession. For binder/motif work, trim the input to the region you actually want to build
against — smaller inputs are kinder on VRAM and easier to reason about.


In [ ]:
#@title Load target (upload or fetch)
import os, urllib.request
SOURCE = "upload"  #@param ["upload", "fetch_pdb"]
PDB_ID = "1ABC"    #@param {type:"string"}
WORK = globals().get("WORK", "/content/run"); os.makedirs(WORK, exist_ok=True)
RAW_TARGET = os.path.join(WORK, "target_raw.pdb")

if SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    src = list(up.keys())[0]
    with open(RAW_TARGET, "wb") as f:   # write bytes directly — works across local/Drive
        f.write(up[src])
else:
    url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb"
    urllib.request.urlretrieve(url, RAW_TARGET)

print("Target ->", RAW_TARGET)
import biotite.structure.io as strucio
arr = strucio.load_structure(RAW_TARGET)
import numpy as np
print("chains:", sorted(set(arr.chain_id)), "| residues:",
      len(np.unique(np.stack([arr.chain_id, arr.res_id]), axis=1).T))


## 5 · Symmetric targets only — align axis → z, centre → origin *(skip for non-symmetric runs)*

**Only run this if you're doing symmetric design.** RFD3 symmetric sampling expects the
symmetric motif **pre-symmetrised around the origin** with the symmetry axis on **z**. A raw
cryo-EM/AF3 assembly is usually only approximately symmetric and sits at an arbitrary
position/orientation.

Strategy below:
1. Estimate the Cn axis from the per-chain Cα centres of mass (SVD plane-normal).
2. Translate the ring centre to the origin and rotate the axis onto z.
3. *(Recommended)* Rebuild an **idealised Cn assembly** from a single protomer by applying
   `k·(360/n)°` rotations about z — this guarantees exact symmetry, which RFD3 wants.

Set `RING_CHAINS` to the chains forming your ring, `PROTOMER_CHAIN` to the one you trust most
(best density / highest pLDDT), and `SYM_ORDER` to n. Writes `target_sym.pdb`.


In [ ]:
#@title Align & idealise the symmetric target
import numpy as np, biotite.structure as struc, biotite.structure.io as strucio

RING_CHAINS    = "A,B,C,D,E,F"  #@param {type:"string"}
PROTOMER_CHAIN = "A"            #@param {type:"string"}
SYM_ORDER      = 6              #@param {type:"integer"}
IDEALISE       = True           #@param {type:"boolean"}
ALIGNED_TARGET = f"{globals().get('WORK','/content/run')}/target_sym.pdb"

ring = [c.strip() for c in RING_CHAINS.split(",") if c.strip()]
arr  = strucio.load_structure(f"{globals().get('WORK','/content/run')}/target_raw.pdb")

# per-chain Cα COM
def chain_ca_com(a, ch):
    m = (a.chain_id == ch) & (a.atom_name == "CA")
    return a.coord[m].mean(0)
coms = np.stack([chain_ca_com(arr, c) for c in ring])
centre = coms.mean(0)

# Cn axis = plane normal of the COM ring (smallest singular vector)
_, _, Vt = np.linalg.svd(coms - centre)
axis = Vt[-1] / np.linalg.norm(Vt[-1])

def rot_a_to_b(a, b):
    a = a/np.linalg.norm(a); b = b/np.linalg.norm(b)
    v = np.cross(a, b); c = float(np.dot(a, b))
    if np.linalg.norm(v) < 1e-8:
        return np.eye(3) if c > 0 else -np.eye(3)
    vx = np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
    return np.eye(3) + vx + vx@vx*(1/(1+c))

R = rot_a_to_b(axis, np.array([0.,0.,1.]))
arr.coord = (arr.coord - centre) @ R.T          # centre + align axis to z

def rotz(deg):
    t = np.deg2rad(deg); c,s = np.cos(t), np.sin(t)
    return np.array([[c,-s,0],[s,c,0],[0,0,1.]])

if IDEALISE:
    proto = arr[arr.chain_id == PROTOMER_CHAIN].copy()
    labels = [chr(ord("A")+i) for i in range(SYM_ORDER)]
    subs = []
    for i in range(SYM_ORDER):
        s = proto.copy()
        s.coord = s.coord @ rotz(360.0/SYM_ORDER*i).T
        s.chain_id = np.full(s.array_length(), labels[i])
        subs.append(s)
    out = subs[0]
    for s in subs[1:]:
        out += s
else:
    out = arr[np.isin(arr.chain_id, ring)].copy()

strucio.save_structure(ALIGNED_TARGET, out)
print("Wrote", ALIGNED_TARGET, "| chains:", sorted(set(out.chain_id)))
print("Cn axis (raw frame):", np.round(axis,3), "| ring centre:", np.round(centre,2))


## 6 · Prepare target — strip ligands / trim residues  *(do this for large complexes)*

This is the cell that keeps RFD3 in memory. The model featurises **everything** in the input
(atomised ligands and full-length chains included), and pairwise cost scales as O(N²) in tokens,
so a full multi-protomer assembly with bound nucleotide will OOM even on an A100. Trim the input
to the chains and residue window you actually design against, and drop non-polymer atoms (ATP,
ions, waters) unless you specifically need them.

- **Symmetric run:** keep all ring chains (`A,B,C,D,E,F`) and trim to your epitope window — that
  gives the small pre-symmetrised motif the C6 sampler replicates.
- **Asymmetric / multi-protomer run:** keep just the 2–3 protomers around your site, trimmed.
- **Keeping a ligand:** leave `STRIP_NONPOLYMER` on for now; the clean way to retain a nucleotide
  is via RFD3's `ligand` field, not by letting it ride along as unnamed hetero.


In [ ]:
#@title Prepare / trim target
import os, numpy as np, biotite.structure as struc, biotite.structure.io as strucio

SOURCE_PDB       = "auto"             #@param {type:"string"}
KEEP_CHAINS      = "A,B,C,D,E,F"      #@param {type:"string"}
RES_LO           = 0                   #@param {type:"integer"}
RES_HI           = 0                   #@param {type:"integer"}
STRIP_NONPOLYMER = True                #@param {type:"boolean"}
# SOURCE_PDB "auto" = target_sym.pdb if it exists (symmetric), else target_raw.pdb
# RES_LO/RES_HI both 0 = no residue trim. KEEP_CHAINS "" = all chains.

WORK = globals().get("WORK", "/content/run")
if SOURCE_PDB == "auto":
    SOURCE_PDB = (f"{WORK}/target_sym.pdb" if os.path.exists(f"{WORK}/target_sym.pdb")
                  else f"{WORK}/target_raw.pdb")
PREPARED = f"{WORK}/target_prepared.pdb"

arr  = strucio.load_structure(SOURCE_PDB)
mask = np.ones(arr.array_length(), bool)
if STRIP_NONPOLYMER:
    mask &= struc.filter_amino_acids(arr)          # drops ATP / ions / waters
if KEEP_CHAINS.strip():
    mask &= np.isin(arr.chain_id, [c.strip() for c in KEEP_CHAINS.split(",") if c.strip()])
if RES_LO and RES_HI:
    mask &= (arr.res_id >= RES_LO) & (arr.res_id <= RES_HI)

arr = arr[mask]
strucio.save_structure(PREPARED, arr)
n_res = len(np.unique(np.stack([arr.chain_id, arr.res_id]), axis=1).T)
print(f"Source : {SOURCE_PDB}")
print(f"Wrote  : {PREPARED}")
print(f"Kept   : {arr.array_length()} atoms | {n_res} residues | chains {sorted(set(arr.chain_id))}")
print("\\nUse this path as INPUT_PDB in the next cell.")


## 7 · Configure the design run

Build the RFD3 input JSON. Common patterns
([input spec](https://rosettacommons.github.io/foundry/models/rfd3/input.html),
[symmetry](https://rosettacommons.github.io/foundry/models/rfd3/examples/symmetry.html),
[PPI tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/ppi_design_tutorial.html)):

- **Unconditional / monomer** — leave `INPUT_PDB` empty and set `BINDER_LENGTH` to a single
  number or range (the spec falls back to `length`-only with no input).
- **Binder / motif scaffolding** — point `INPUT_PDB` at your prepared target
  (`/content/run/target_prepared.pdb` from step 6), set `CONTIG` to keep the
  target (`<chain><range>`), a chain break (`/0`), then a designed segment (e.g. `60-100`).
  Add `select_hotspots` to steer the interface (atoms ≤ ~4.5 Å; residues must appear in the contig).
- **Symmetric** — set `USE_SYMMETRY=True` and `SYM_ID` (e.g. `C6`, `D2`; only C/D supported),
  point `INPUT_PDB` at the prepared symmetric file from step 6, and write `CONTIG` as one
  asymmetric unit (RFD3 replicates it n-fold).

`infer_ori_strategy: hotspots` places the design COM ~10 Å out from the hotspot COM;
`is_non_loopy: true` biases toward helices (good for PPI). The cell prints the JSON for
hand-editing.


In [ ]:
#@title Build input.json
import json, os

DESIGN_NAME    = "design"                   #@param {type:"string"}
INPUT_PDB      = ""                          #@param {type:"string"}
CONTIG         = ""                          #@param {type:"string"}
BINDER_LENGTH  = "60-100"                    #@param {type:"string"}
HOTSPOTS       = ""                          #@param {type:"string"}
IS_NON_LOOPY   = True                        #@param {type:"boolean"}
INFER_ORI      = "hotspots"                  #@param ["hotspots", "com", "none"]
USE_SYMMETRY   = False                       #@param {type:"boolean"}
SYM_ID         = "C6"                        #@param {type:"string"}

# INPUT_PDB: typically /content/run/target_prepared.pdb  (from step 6); empty = unconditional
# HOTSPOTS: residue-range string "A230-233,A235-236"  OR  atom-level "A232:ALL; A235:CA,CB"
# CONTIG examples:  "A1-300,/0,60-100"  (binder)   |   leave empty for length-only
# INFER_ORI "none" omits infer_ori_strategy (use for central plugs that converge on the axis)

spec = {"is_non_loopy": IS_NON_LOOPY}

if INPUT_PDB.strip():
    spec["input"] = INPUT_PDB.strip()
    if CONTIG.strip():
        spec["contig"] = CONTIG.strip()
    if INFER_ORI != "none":
        spec["infer_ori_strategy"] = INFER_ORI
else:
    # unconditional / monomer: no input → length only. Hotspots/contig need a target.
    if CONTIG.strip() or HOTSPOTS.strip():
        raise ValueError(
            "CONTIG / HOTSPOTS require an INPUT_PDB (the target structure). "
            "Set INPUT_PDB (e.g. /content/run/target_prepared.pdb), or clear CONTIG and HOTSPOTS "
            "for an unconditional/monomer run.")
    spec["length"] = BINDER_LENGTH

if HOTSPOTS.strip():
    hs = HOTSPOTS.strip()
    if ":" in hs:
        # atom/residue-level dict form:  "A232:ALL; A235:CA,CB"
        hot = {}
        for tok in hs.split(";"):
            tok = tok.strip()
            if not tok: continue
            res, atoms = tok.split(":")
            hot[res.strip()] = atoms.strip()
        spec["select_hotspots"] = hot
    else:
        # residue-level contig-string form:  "A230-233,A235-236"
        spec["select_hotspots"] = hs

if USE_SYMMETRY:
    spec["symmetry"] = {"id": SYM_ID, "is_symmetric_motif": True}

# sanity echo
print("input set :", "input" in spec)
print("contig    :", spec.get("contig", "(none)"))
print("hotspots  :", spec.get("select_hotspots", "(none)"))
print("symmetry  :", spec.get("symmetry", "(none)"))
print("-" * 40)

inp = {DESIGN_NAME: spec}
WORK = globals().get("WORK", "/content/run")
IN_JSON = f"{WORK}/input.json"
os.makedirs(WORK, exist_ok=True)
with open(IN_JSON, "w") as f:
    json.dump(inp, f, indent=2)
print(json.dumps(inp, indent=2))


## 8 · Run RFD3 (backbone generation)

For symmetry, `inference_sampler.kind=symmetry` and `diffusion_batch_size=1` are required;
add `low_memory_mode=True` on a T4. `n_batches` × `diffusion_batch_size` = total designs
(symmetry forces batch size 1, so scale designs with `n_batches`). `num_timesteps=200` is the
default — drop it to ~50 for a fast smoke test, raise for quality.


In [ ]:
#@title RFD3 design
import os
N_DESIGNS      = 8     #@param {type:"integer"}
NUM_TIMESTEPS  = 200   #@param {type:"integer"}
STEP_SCALE     = 1.5   #@param {type:"number"}
LOW_MEMORY     = True  #@param {type:"boolean"}
USE_SYMMETRY   = False #@param {type:"boolean"}

WORK = globals().get("WORK", "/content/run")
RFD3_OUT = f"{WORK}/rfd3_out"
os.environ.setdefault("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduce fragmentation OOMs

INPUT_JSON = f"{WORK}/input.json"
cmd = [
    "rfd3 design",
    f"out_dir={RFD3_OUT}",
    f"inputs={INPUT_JSON}",
    "prevalidate_inputs=True",
    "skip_existing=False",
    f"n_batches={N_DESIGNS}",
    "diffusion_batch_size=1" if USE_SYMMETRY else "diffusion_batch_size=8",
    f"inference_sampler.num_timesteps={NUM_TIMESTEPS}",
    f"inference_sampler.step_scale={STEP_SCALE}",
    f"low_memory_mode={LOW_MEMORY}",
]
if USE_SYMMETRY:
    cmd.append("inference_sampler.kind=symmetry")

full = " ".join(cmd)
print(full, "\\n")
get_ipython().system(full)

import glob
backbones = sorted(glob.glob(f"{RFD3_OUT}/**/*.cif.gz", recursive=True))
print(f"\\n{len(backbones)} backbone(s):")
for b in backbones: print(" ", b)


## 9 · LigandMPNN (sequence design)

Redesign each RFD3 backbone's sequence. Unlike `rfd3`/`rf3`, `mpnn` does **not** auto-discover
its weights — it requires an explicit `--checkpoint_path` plus `--is_legacy_weights`
(the cell auto-locates the checkpoint under `FOUNDRY_CHECKPOINT_DIRS`).

Two controls that matter for binder design:
- **`FIXED_CHAINS`** — keep your target chains fixed so MPNN only designs the binder, not the
  target's real sequence. Set this to your target chain IDs (e.g. `A,D`).
- **`TIE_CHAINS`** *(symmetric runs only)* — list the binder subunit chains to tie via
  `--homo_oligomer_chains`, so all subunits get one shared sequence (required for a symmetric
  cap; otherwise each subunit is designed independently and the symmetry breaks).

`--is_legacy_weights` defaults to `True` (matches the Foundry tutorial for installed weights);
flip to `False` if MPNN errors or the sequences look wrong.


In [ ]:
#@title LigandMPNN over all backbones
import glob, os, subprocess, json as _json
MODEL_TYPE     = "ligand_mpnn"  #@param ["ligand_mpnn", "protein_mpnn"]
IS_LEGACY      = True           #@param {type:"boolean"}
FIXED_CHAINS   = ""             #@param {type:"string"}
TIE_CHAINS     = ""             #@param {type:"string"}
N_SEQUENCES    = 4              #@param {type:"integer"}
TEMPERATURE    = 0.1            #@param {type:"number"}
# FIXED_CHAINS: target chains to keep fixed, e.g. "A,D"  (design only the binder)
# TIE_CHAINS  : symmetric runs only — binder subunit chains to tie to one sequence, e.g. "G,H,I,J,K,L"

CKPT_DIR = os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")
WORK     = globals().get("WORK", "/content/run")
RFD3_OUT = f"{WORK}/rfd3_out"
MPNN_OUT = f"{WORK}/mpnn_out"; os.makedirs(MPNN_OUT, exist_ok=True)

# mpnn needs an explicit checkpoint path — locate it
ckpts = [p for p in glob.glob(f"{CKPT_DIR}/**/*", recursive=True)
         if os.path.isfile(p) and "mpnn" in p.lower()
         and p.endswith((".pt", ".ckpt", ".pth", ".safetensors"))]
assert ckpts, f"No MPNN checkpoint under {CKPT_DIR}; run step 3. Found nothing matching *mpnn*."
# prefer a ligand-named checkpoint when using ligand_mpnn
MPNN_CKPT = sorted(ckpts, key=lambda p: ("ligand" not in p.lower()))[0]
print("MPNN checkpoint:", MPNN_CKPT)

backbones = (sorted(glob.glob(f"{RFD3_OUT}/**/*.cif.gz", recursive=True))
             or sorted(glob.glob(f"{RFD3_OUT}/**/*.cif", recursive=True))
             or sorted(glob.glob(f"{RFD3_OUT}/**/*.pdb", recursive=True)))
print(f"{len(backbones)} backbone(s)\n")

for bb in backbones:
    cmd = ["mpnn",
           "--model_type", MODEL_TYPE,
           "--checkpoint_path", MPNN_CKPT,
           "--is_legacy_weights", str(IS_LEGACY),
           "--structure_path", bb,
           "--out_directory", MPNN_OUT,
           "--batch_size", str(N_SEQUENCES),
           "--temperature", str(TEMPERATURE)]
    if FIXED_CHAINS.strip():
        cmd += ["--fixed_chains", FIXED_CHAINS.strip()]
    if TIE_CHAINS.strip():
        chains = [c.strip() for c in TIE_CHAINS.split(",") if c.strip()]
        cmd += ["--homo_oligomer_chains", _json.dumps([chains])]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(os.path.basename(bb), "->", "ok" if r.returncode == 0 else f"FAIL rc={r.returncode}")
    if r.returncode != 0:
        print(r.stderr[-1500:]); break   # stop on first failure so the error is visible

cifs = sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True))
fas  = sorted(glob.glob(f"{MPNN_OUT}/**/*.fa*", recursive=True))
print(f"\\n{len(cifs)} CIF + {len(fas)} FASTA in {MPNN_OUT}")


## 10 · RF3 (refold / validation)

Refold each MPNN-designed structure with RF3 and keep its confidence metrics — this is your
AF3-style designability check. Pass **absolute paths** to the input CIFs.


In [ ]:
#@title RF3 fold
import glob, os, subprocess
NUM_STEPS = 200  #@param {type:"integer"}

WORK = globals().get("WORK", "/content/run")
MPNN_OUT = f"{WORK}/mpnn_out"
RF3_OUT  = f"{WORK}/rf3_out"; os.makedirs(RF3_OUT, exist_ok=True)
os.environ.setdefault("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry_ckpts")

cifs = sorted(glob.glob(f"{MPNN_OUT}/**/*.cif", recursive=True))
print(f"{len(cifs)} MPNN structure(s) to refold\n")

for c in cifs:
    cmd = ["rf3", "fold",
           f"inputs={os.path.abspath(c)}",
           f"out_dir={RF3_OUT}",
           f"num_steps={NUM_STEPS}",
           "diffusion_batch_size=1"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(os.path.basename(c), "->", "ok" if r.returncode == 0 else f"FAIL rc={r.returncode}")
    if r.returncode != 0:
        print(r.stderr[-1500:]); break

preds = sorted(glob.glob(f"{RF3_OUT}/**/*.cif*", recursive=True))
print(f"\\n{len(preds)} RF3 prediction(s) in {RF3_OUT}")


## 11 · Collect metrics & rank designs

Pulls confidence fields (pLDDT / pTM / pAE-type) from the RF3 sidecar JSONs and tabulates
them. Field names vary by version, so this scans each JSON for likely keys rather than
assuming a fixed schema. For a true binder filter you'll add interface metrics
(interface pAE / ipTM, BSA, contact counts) — hooks noted in the cell.


In [ ]:
#@title Build metrics table
import glob, json, os, pandas as pd

WORK = globals().get("WORK", "/content/run")
RF3_OUT = f"{WORK}/rf3_out"
jsons = sorted(glob.glob(f"{RF3_OUT}/**/*.json", recursive=True))

WANT = ["plddt", "mean_plddt", "ptm", "iptm", "pae", "mean_pae", "confidence"]
rows = []
for j in jsons:
    try:
        d = json.load(open(j))
    except Exception:
        continue
    flat = {}
    def walk(prefix, obj):
        if isinstance(obj, dict):
            for k, v in obj.items(): walk(f"{prefix}{k}.", v)
        elif isinstance(obj, (int, float)):
            flat[prefix[:-1]] = obj
    walk("", d)
    row = {"design": os.path.basename(j)}
    for k, v in flat.items():
        if any(w in k.lower() for w in WANT):
            row[k] = v
    rows.append(row)

df = pd.DataFrame(rows)
if not df.empty:
    sort_col = next((c for c in df.columns if "plddt" in c.lower()), None)
    if sort_col: df = df.sort_values(sort_col, ascending=False)
    df.to_csv(f"{WORK}/design_metrics.csv", index=False)
print(df.head(20).to_string() if not df.empty else "No JSON metrics parsed — inspect rf3_out/ manually.")

# --- TODO: interface metrics for binder filtering -----------------------------
# Compute on the RF3 complex: interface pAE (binder<->target blocks of the PAE matrix),
# buried surface area (biotite SASA: complex vs isolated chains), and Cα contact counts
# within 8 Å across the interface. These map onto the ipSAE/BSA filters you already use.


## 12 · Download results

In [ ]:
#@title Zip & download outputs
import shutil, os
from google.colab import files
WORK = globals().get("WORK", "/content/run")

if WORK.startswith("/content/drive/"):
    print("Outputs already persisted on Drive at:", WORK, "\\n(no download needed)")
else:
    shutil.make_archive("/content/rfd3_run", "zip", WORK)
    files.download("/content/rfd3_run.zip")


---
### Notes & next steps
- **Symmetry sanity check.** If you ran symmetric, open a backbone in PyMOL/ChimeraX and
  confirm it's genuinely Cn/Dn and seated on your hotspots. Drift off-axis usually means
  revisiting step 5 (alignment / idealisation) before touching diffusion params.
- **Designability loop.** The real filter is RF3-refold vs RFD3-backbone agreement (Cα-RMSD)
  plus interface confidence — wire the RMSD + interface-pAE/BSA hooks in step 11 into your
  filtering.
- **Throughput.** Symmetric runs are batch-1; scale with `n_batches` or move off the T4. For
  large screens this is better as a SLURM array than Colab.
- **Docs:** [input spec](https://rosettacommons.github.io/foundry/models/rfd3/input.html) ·
  [symmetry](https://rosettacommons.github.io/foundry/models/rfd3/examples/symmetry.html) ·
  [PPI tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/ppi_design_tutorial.html) ·
  [Foundry repo](https://github.com/RosettaCommons/foundry).
